# 06 · Placar e erros

Placar das 3 configurações oficiais a partir de `avaliacao/resultados/*.jsonl`:

1. **sem_contexto** (config 1): só o LLM, sem instrução de recusa.
2. **rag** (config 2): top-3 + duas barreiras de recusa. **O RAG não tem como responder rota** (não tem o GTFS): a falsa recusa de rota dele é esperada.
3. **agente** (config 3): roteador + ferramentas (`buscar_base`, `consultar_gtfs`, `previsao_onibus`).

A config 1b (com instrução de recusa) fica numa tabela à parte, de apêndice. Mesmo modelo (`qwen2.5:3b`, local), mesmos 63 casos, mesmos critérios de acerto.

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src')); sys.path.insert(0, str(RAIZ / 'avaliacao'))
import avaliar as av

def carregar(nome):
    return [json.loads(l) for l in (RAIZ / 'avaliacao' / 'resultados' / f'{nome}.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

ROTULO = {'sem_contexto': '1 sem_contexto', 'rag': '2 rag', 'agente': '3 agente'}
res = {n: carregar(n) for n in ROTULO}
apendice = carregar('sem_contexto_instr')
casos = {c['id']: c for c in av.carregar_casos()}
print({n: len(r) for n, r in res.items()}, '| 1b:', len(apendice))

{'sem_contexto': 63, 'rag': 63, 'agente': 63} | 1b: 63


## 1. Placar geral (63 casos)

`acerto` e `alucinação` valem sobre todos os casos; `recusa correta` só sobre os 15 `nao_consta`; `falsa recusa` sobre os casos que tinham resposta (30 `texto` + as rotas com linha esperada); `citação correta` só sobre os 30 `texto`.

In [2]:
cols = ['acuracia', 'alucinacao', 'recusa_correta', 'falsa_recusa', 'json_valido', 'citacao_correta', 'latencia_media_s']
geral = pd.DataFrame({ROTULO[n]: av.resumo(r) for n, r in res.items()}).T[cols]
geral.columns = ['acerto', 'alucinação', 'recusa correta', 'falsa recusa', 'JSON válido', 'citação correta', 'latência média (s)']
print(geral.to_string())

                acerto  alucinação  recusa correta  falsa recusa  JSON válido  citação correta  latência média (s)
1 sem_contexto  0.2381      0.7619          0.0667        0.0000          1.0           0.0000                0.74
2 rag           0.6825      0.0952          0.9333        0.3333          1.0           0.8000                2.87
3 agente        0.7460      0.1587          0.9333        0.1429          1.0           0.6667                1.87


## 2. Acerto e falsa recusa por tipo

**Atenção ao RAG em `rota`:** ele não consulta o GTFS, então só pode recusar (falsa recusa de 100% é esperada, não é falha de implementação). O acerto de rota dele vem só dos 6 casos sem linha esperada, em que recusar é a resposta certa.

In [3]:
linhas = []
for tipo in ('texto', 'rota', 'nao_consta'):
    for n, r in res.items():
        s = av.resumo([x for x in r if x['tipo'] == tipo])
        linhas.append({'tipo': tipo, 'config': ROTULO[n], 'casos': s['n'], 'acerto': s['acuracia'], 'alucinação': s['alucinacao'],
                       'falsa recusa': s['falsa_recusa'], 'recusa correta': s['recusa_correta']})
por_tipo = pd.DataFrame(linhas)
print(por_tipo.to_string(index=False, na_rep='-'))
print()
rota_com_linha = [x for x in res['rag'] if x['tipo'] == 'rota' and casos[x['id']]['linhas_esperadas']]
print(f"RAG em rota com linha esperada: {len(rota_com_linha)} casos, recusou {sum(x['saida']['recusou'] for x in rota_com_linha)} (nenhum acerto possível sem GTFS)")

      tipo         config  casos  acerto  alucinação  falsa recusa  recusa correta
     texto 1 sem_contexto     30  0.3667      0.6333        0.0000               -
     texto          2 rag     30  0.7667      0.1667        0.0667               -
     texto       3 agente     30  0.6667      0.1667        0.1667               -
      rota 1 sem_contexto     18  0.1667      0.8333        0.0000               -
      rota          2 rag     18  0.3333      0.0000        1.0000               -
      rota       3 agente     18  0.7222      0.2222        0.0833               -
nao_consta 1 sem_contexto     15  0.0667      0.9333             -          0.0667
nao_consta          2 rag     15  0.9333      0.0667             -          0.9333
nao_consta       3 agente     15  0.9333      0.0667             -          0.9333

RAG em rota com linha esperada: 12 casos, recusou 12 (nenhum acerto possível sem GTFS)


## 3. Gráfico de barras agrupadas → `avaliacao/placar.png`

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# cores: slots 1-3 da paleta categórica validada (azul, laranja, aqua); a aqua tem contraste < 3:1,
# então todas as barras levam o valor escrito (rótulo visível) e há legenda.
CORES = {'1 sem_contexto': '#2a78d6', '2 rag': '#eb6834', '3 agente': '#1baf7a'}
metricas = [('acerto', 'acuracia', '↑'), ('alucinação', 'alucinacao', '↓'), ('recusa correta', 'recusa_correta', '↑'),
            ('falsa recusa', 'falsa_recusa', '↓'), ('JSON válido', 'json_valido', '↑'), ('citação correta', 'citacao_correta', '↑')]
resumos = {ROTULO[n]: av.resumo(r) for n, r in res.items()}

fig, ax = plt.subplots(figsize=(11, 4.8))
largura = 0.25
x = np.arange(len(metricas))
for i, (cfg, cor) in enumerate(CORES.items()):
    valores = [resumos[cfg][chave] or 0 for _, chave, _ in metricas]
    barras = ax.bar(x + (i - 1) * (largura + 0.02), valores, largura, label=cfg, color=cor, edgecolor='#fcfcfb', linewidth=2)
    for b, v in zip(barras, valores):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.2f}', ha='center', va='bottom', fontsize=8, color='#0b0b0b')
ax.set_xticks(x)
ax.set_xticklabels([f'{nome} {seta}' for nome, _, seta in metricas], fontsize=10, color='#0b0b0b')
ax.set_ylim(0, 1.12); ax.set_ylabel('proporção dos casos do grupo', color='#52514e')
ax.set_title('Placar das 3 configurações (63 casos, qwen2.5:3b local)  ·  ↑ maior é melhor, ↓ menor é melhor', fontsize=11, color='#0b0b0b', loc='left')
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
ax.set_facecolor('#fcfcfb'); fig.patch.set_facecolor('#fcfcfb')
ax.grid(axis='y', color='#e1e0d9', linewidth=0.8); ax.set_axisbelow(True)
for lado in ('top', 'right', 'left'):
    ax.spines[lado].set_visible(False)
ax.spines['bottom'].set_color('#c3c2b7'); ax.tick_params(colors='#898781', length=0)
fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'placar.png', dpi=140, facecolor=fig.get_facecolor())
plt.close(fig)
print('salvo: avaliacao/placar.png')

salvo: avaliacao/placar.png


## 4. Roteador do agente: tipo do caso × ferramenta escolhida → `avaliacao/roteador.png`

In [5]:
ferramentas = ['buscar_base', 'consultar_gtfs', 'previsao_onibus', 'nenhuma']
tipos = ['texto', 'rota', 'nao_consta']
matriz = pd.crosstab(pd.Series([r['tipo'] for r in res['agente']], name='tipo'),
                     pd.Series([r['saida']['ferramenta'] for r in res['agente']], name='ferramenta')).reindex(index=tipos, columns=ferramentas, fill_value=0)
print(matriz.to_string())

fig, ax = plt.subplots(figsize=(8.6, 4.0))
azul = matplotlib.colors.LinearSegmentedColormap.from_list('azul', ['#cde2fb', '#2a78d6', '#0d366b'])
im = ax.imshow(matriz.values, cmap=azul, vmin=0, vmax=matriz.values.max())
ax.set_xticks(range(len(ferramentas))); ax.set_xticklabels([f.replace('_', '\n') for f in ferramentas], fontsize=10)
ax.set_yticks(range(len(tipos))); ax.set_yticklabels([f'{t}\n({int(matriz.loc[t].sum())} casos)' for t in tipos], fontsize=10)
esperado = {('texto', 'buscar_base'), ('rota', 'consultar_gtfs')}
for i, t in enumerate(tipos):
    for j, f in enumerate(ferramentas):
        v = matriz.values[i, j]
        cor = 'white' if v > matriz.values.max() * 0.55 else '#0b0b0b'
        ax.text(j, i, str(v), ha='center', va='center', fontsize=12, color=cor, fontweight='bold' if (t, f) in esperado else 'normal')
ax.set_title('Roteador: ferramenta escolhida por tipo de caso\n(negrito = o esperado; nao_consta: qualquer ferramenta, desde que recuse)', fontsize=10, loc='center')
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'roteador.png', dpi=140, facecolor='white')
plt.close(fig)
certo = sum(r['saida']['ferramenta'] == 'buscar_base' for r in res['agente'] if r['tipo'] == 'texto') + \
        sum(r['saida']['ferramenta'] == 'consultar_gtfs' for r in res['agente'] if r['tipo'] == 'rota') + \
        sum(r['saida']['recusou'] for r in res['agente'] if r['tipo'] == 'nao_consta')
print(f'acurácia do roteador: {certo}/{len(res["agente"])} = {certo/len(res["agente"]):.3f}')
print('salvo: avaliacao/roteador.png')

ferramenta  buscar_base  consultar_gtfs  previsao_onibus  nenhuma
tipo                                                             
texto                27               0                0        3
rota                  0              16                0        2
nao_consta            9               0                1        5
acurácia do roteador: 57/63 = 0.905
salvo: avaliacao/roteador.png


## 5. Apêndice: config 1b (LLM sem contexto, com instrução de recusa)

In [6]:
ap = pd.DataFrame({'1b sem_contexto_instr': av.resumo(apendice)}).T[cols]
ap.columns = geral.columns
print(ap.to_string())
print('\nSem a instrução (config 1) o modelo quase nunca recusa; com ela (1b) recusa quase tudo que não sabe, mas também 40% do que tinha resposta.')

                       acerto  alucinação  recusa correta  falsa recusa  JSON válido  citação correta  latência média (s)
1b sem_contexto_instr  0.4603      0.2698          0.9333        0.4048          1.0              0.0                0.63

Sem a instrução (config 1) o modelo quase nunca recusa; com ela (1b) recusa quase tudo que não sabe, mas também 40% do que tinha resposta.


## 6. Os erros selecionados, nas saídas reais

Os 5 erros (e a extração da `rota_10`, como extra) estão descritos em `avaliacao/erros.md`, seção "5 erros selecionados". Aqui, a saída de cada configuração nesses casos.

In [7]:
selecionados = {
    '1 busca': ['texto_19'], '2 id inventado': ['nao_consta_01'], '3 inversão de "exclusivo"': ['texto_30'],
    '4 falsos positivos do GTFS': ['rota_04', 'rota_08', 'rota_12'], '5 roteador -> nenhuma': ['texto_23', 'texto_25', 'texto_29', 'rota_06', 'rota_14'],
    'extra: extração': ['rota_10'],
}
por_id = {n: {r['id']: r for r in rs} for n, rs in res.items()}
for erro, ids in selecionados.items():
    print('=' * 100); print('ERRO', erro)
    for i in ids:
        print(f"\n{i}: {casos[i]['pergunta']}")
        for n in ROTULO:
            r = por_id[n][i]; s = r['saida']
            print(f"   {ROTULO[n]:<15} acerto={r['metricas']['acerto']!s:<5} recusou={s['recusou']!s:<5} {s['resposta'][:150]}")

ERRO 1 busca

texto_19: quando eu for guiado por alguem, em que braço devo segurar?
   1 sem_contexto  acerto=True  recusou=False Você deve segurar o braço do guia no braço oposto ao seu para melhor equilíbrio.
   2 rag           acerto=False recusou=False Se você for guiado por alguém, segure no braço do guia para manter o contato seguro e seguro durante a viagem.
   3 agente        acerto=False recusou=False Se você for guiado por alguém, segure no braço do guia para manter o contato seguro e seguro durante a viagem.
ERRO 2 id inventado

nao_consta_01: a estação faria lima da linha 4 amarela tem elevador?
   1 sem_contexto  acerto=False recusou=False Não, a estação Lima da Linha 4-Amarela não possui elevador.
   2 rag           acerto=False recusou=False Sim, a estação tem acesso à plataforma por elevador.
   3 agente        acerto=False recusou=False Sim, a estação tem acesso à plataforma por elevador.
ERRO 3 inversão de "exclusivo"

texto_30: uma pessoa cega pode usar o táxi acessi

## 7. Correções medidas: `agente_v2` (C1 a C3), sem alterar o `agente` original

- **C1 (`gtfs.py`, modo `v2`):** casa só `stop_name` (não `stop_desc`); um local casa com o **grupo** de paradas mais denso (paradas casadas encadeadas a ≤ 300 m), com prioridade para nomes com "Terminal"/"Estação" quando a pergunta tiver a palavra. **Ajuste feito durante a implementação:** só "grupo mais denso" escolhia a avenida em vez da estação ("Av. Jabaquara, 79" × estação Jabaquara) e quebrava o teste `Tucuruvi -> Jabaquara`; por isso, entre grupos, vem primeiro o que tem uma parada de nome **igual** ao pedido, e só depois o mais denso. O modo `v1` (padrão) continua igual ao original; os testes valem para os dois modos.
- **C2 (barreira 3):** `citacao_invalida` → recusa.
- **C3 (fallback do roteador):** se o roteador escolher "nenhuma" mas o top-1 da busca for ≥ limiar (0,8537), usa `buscar_base`.

Gabarito dos casos: **não foi alterado**.

In [8]:
from sonar_ia import agente as ag, configs
ag.gtfs(); ag.gtfs(True); configs.buscador()
v2 = av.avaliar('agente_v2', ag.agente_v2, list(casos.values()), verbose=False)
resultados_v2 = {r['id']: r for r in v2}
antes = {r['id']: r for r in res['agente']}
cmp = {'3 agente (original)': res['agente'], '3b agente_v2': v2}
print('GERAL'); print(pd.DataFrame({k: av.resumo(v) for k, v in cmp.items()}).T[cols].to_string())
for t in ('texto', 'rota', 'nao_consta'):
    print(f'\n{t.upper()}')
    print(pd.DataFrame({k: av.resumo([r for r in v if r['tipo'] == t]) for k, v in cmp.items()}).T[cols].to_string())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

GERAL
                     acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
3 agente (original)    0.7460      0.1587          0.9333        0.1429          1.0           0.6667              1.87
3b agente_v2           0.8413      0.0952          1.0000        0.0952          1.0           0.8000              3.57

TEXTO
                     acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
3 agente (original)    0.6667      0.1667             NaN        0.1667          1.0           0.6667              2.03
3b agente_v2           0.7333      0.2000             NaN        0.0667          1.0           0.8000              5.51

ROTA
                     acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
3 agente (original)    0.7222      0.2222             NaN        0.0833          1.0              NaN               2.3
3b agente_v2         

### Tabela: erro selecionado × antes × depois

In [9]:
grupos = [
    ('1 busca (texto_19)', ['texto_19']), ('2 id inventado (nao_consta_01)', ['nao_consta_01']),
    ('3 inversão de "exclusivo" (texto_30)', ['texto_30']), ('4 falsos positivos do GTFS', ['rota_04', 'rota_08', 'rota_12']),
    ('5 roteador -> nenhuma', ['texto_23', 'texto_25', 'texto_29', 'rota_06', 'rota_14']), ('extra: extração (rota_10)', ['rota_10']),
]
def curta(r):
    s = r['saida']
    return ('[recusa] ' if s['recusou'] else '') + s['resposta'].replace('\n', ' ')[:62] + f" ({s['ferramenta']})"
linhas = []
for nome, ids in grupos:
    for i in ids:
        a, d = antes[i], resultados_v2[i]
        ma, md = a['metricas']['acerto'], d['metricas']['acerto']
        linhas.append({'erro': nome, 'caso': i, 'antes': 'ACERTA' if ma else 'erra', 'depois': 'ACERTA' if md else 'erra',
                       'resultado': 'corrigido' if (md and not ma) else 'igual' if ma == md else 'PIOROU',
                       'resposta antes': curta(a), 'resposta depois': curta(d)})
tab = pd.DataFrame(linhas)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 80)
print(tab[['erro', 'caso', 'antes', 'depois', 'resultado']].to_string(index=False))
print()
print(tab['resultado'].value_counts().to_string())
print('\nRespostas:')
for l in linhas:
    print(f"  {l['caso']:<14} antes: {l['resposta antes']}\n  {'':<14} depois: {l['resposta depois']}")

                                erro          caso  antes depois resultado
                  1 busca (texto_19)      texto_19   erra   erra     igual
      2 id inventado (nao_consta_01) nao_consta_01   erra ACERTA corrigido
3 inversão de "exclusivo" (texto_30)      texto_30   erra   erra     igual
          4 falsos positivos do GTFS       rota_04   erra ACERTA corrigido
          4 falsos positivos do GTFS       rota_08   erra ACERTA corrigido
          4 falsos positivos do GTFS       rota_12   erra ACERTA corrigido
               5 roteador -> nenhuma      texto_23   erra ACERTA corrigido
               5 roteador -> nenhuma      texto_25   erra ACERTA corrigido
               5 roteador -> nenhuma      texto_29   erra ACERTA corrigido
               5 roteador -> nenhuma       rota_06   erra   erra     igual
               5 roteador -> nenhuma       rota_14 ACERTA ACERTA     igual
           extra: extração (rota_10)       rota_10   erra ACERTA corrigido

resultado
corrigido    8

### Todos os casos em que o acerto mudou (inclui os que não estavam entre os erros selecionados)

In [10]:
for i, a in antes.items():
    ma, md = a['metricas']['acerto'], resultados_v2[i]['metricas']['acerto']
    if ma != md:
        s = resultados_v2[i]['saida']
        print(f"{i:<14} {'ACERTA' if ma else 'erra':<7} -> {'ACERTA' if md else 'erra':<7} {s['ferramenta']:<15} barreira={s['barreira']!s:<17} {s['resposta'][:90]}")
print()
print('citacao_invalida (barreira 3) em v2:', [i for i, r in resultados_v2.items() if r['saida']['barreira'] == 'citacao_invalida'])
print('fallback do roteador (C3) em v2     :', [i for i, r in resultados_v2.items() if r['saida'].get('fallback_roteador')])
print('alucinações do agente_v2:', [i for i, r in resultados_v2.items() if r['metricas']['alucinacao']])

texto_16       ACERTA  -> erra    buscar_base     barreira=None              Sempre que identificar uma pessoa cega no ponto, pare e informe qual a linha que está oper
texto_23       erra    -> ACERTA  buscar_base     barreira=None              Você precisa levar o CPF (original e cópia), RG ou certidão de nascimento (original e cópi
texto_25       erra    -> ACERTA  buscar_base     barreira=None              As frotas de empresas de táxi devem reservar 10% de seus veículos acessíveis à pessoa com 
texto_29       erra    -> ACERTA  buscar_base     barreira=None              35 estações da CPTM já possuem condições de acesso às pessoas com deficiência ou restrição
rota_04        erra    -> ACERTA  consultar_gtfs  barreira=None              Você pode usar o Trem Estudantes da Linha L11 da CPTM ou o Metrô L3 sentido Corinthians - 
rota_08        erra    -> ACERTA  consultar_gtfs  barreira=ferramenta        Não consta na base de conhecimento.
rota_09        ACERTA  -> erra    consultar_gtf

## 8. Validação (`avaliacao/casos_validacao.jsonl`)

Conjunto separado, escrito depois das correções, para medir se a melhora generaliza. Se o arquivo não existir, esta seção é pulada.

In [11]:
caminho_val = RAIZ / 'avaliacao' / 'casos_validacao.jsonl'
if not caminho_val.exists():
    print('avaliacao/casos_validacao.jsonl NÃO existe: validação pulada (aguardando os casos).')
else:
    cval = av.carregar_casos(caminho_val)
    print(len(cval), 'casos de validação')
    ra = av.avaliar('validacao_agente', ag.agente, cval, verbose=False)
    rb = av.avaliar('validacao_agente_v2', ag.agente_v2, cval, verbose=False)
    for nome, regs in (('agente (original)', ra), ('agente_v2', rb)):
        print('\n' + nome); print(pd.DataFrame({'geral': av.resumo(regs), **av.resumo_por_tipo(regs)}).T[cols].to_string())

12 casos de validação

agente (original)
            acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
geral         0.7500      0.1667             1.0        0.1429          1.0              1.0              2.04
texto         0.7500      0.2500             NaN        0.0000          1.0              1.0              2.48
rota          0.3333      0.3333             NaN        0.3333          1.0              NaN              1.37
nao_consta    1.0000      0.0000             1.0           NaN          1.0              NaN              2.09

agente_v2
            acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
geral         0.7500      0.1667             1.0        0.1429          1.0              1.0              1.78
texto         0.7500      0.2500             NaN        0.0000          1.0              1.0              2.62
rota          0.3333      0.3333             NaN        0.33

## 9. Duas ressalvas de medição

**(a) Variação entre execuções.** O mesmo `agente` (mesmo código, `temperatura=0`, `seed=42`) foi rodado uma segunda vez (`resultados/agente_rerun.jsonl`). O Ollama não é 100% determinístico, e isso entra como ruído nos números.

In [12]:
rerun = carregar('agente_rerun')
orig = {r['id']: r for r in res['agente']}
mudou_texto = [r['id'] for r in rerun if r['saida']['resposta'] != orig[r['id']]['saida']['resposta']]
mudou_acerto = [(r['id'], orig[r['id']]['metricas']['acerto'], r['metricas']['acerto']) for r in rerun if r['metricas']['acerto'] != orig[r['id']]['metricas']['acerto']]
print('texto da resposta diferente na reexecução:', len(mudou_texto), 'de', len(rerun))
print('acerto diferente:', mudou_acerto)
print('acerto geral: original', av.resumo(res['agente'])['acuracia'], '| reexecução', av.resumo(rerun)['acuracia'])
mesmo_caminho = [r['id'] for r in rerun if r['saida']['recuperados'] == orig[r['id']]['saida']['recuperados'] and r['saida']['ferramenta'] == orig[r['id']]['saida']['ferramenta']]
print('mesma ferramenta e mesmos trechos recuperados em', len(mesmo_caminho), 'casos (a diferença vem só da geração do LLM)')

texto da resposta diferente na reexecução: 16 de 63
acerto diferente: [('texto_16', True, False)]
acerto geral: original 0.746 | reexecução 0.7302
mesma ferramenta e mesmos trechos recuperados em 63 casos (a diferença vem só da geração do LLM)


**(b) `recusou=true` com texto que não é uma recusa.** A métrica de recusa lê o campo `recusou`. Se o modelo marca `recusou=true` mas escreve uma resposta, a "recusa correta" é contada, mas o usuário ouve uma resposta (possivelmente inventada).

In [13]:
RECUSAS = {'Não consta na base de conhecimento.', 'Não tenho previsão em tempo real agora.'}
arquivos = ['sem_contexto', 'sem_contexto_instr', 'rag', 'agente', 'agente_v2', 'validacao_agente', 'validacao_agente_v2']
for n in arquivos:
    rs = carregar(n)
    inc = [(r['id'], r['saida']['resposta'][:70]) for r in rs if r['saida']['recusou'] and r['saida']['resposta'] not in RECUSAS]
    print(f'{n:<22} recusou=true com texto de resposta: {inc}')

sem_contexto           recusou=true com texto de resposta: [('nao_consta_11', 'Desculpe, não tenho acesso ao horário real de passagem dos ônibus. Rec')]
sem_contexto_instr     recusou=true com texto de resposta: []
rag                    recusou=true com texto de resposta: [('nao_consta_12', 'Para conseguir um cão-guia, é necessário se inscrever em um centro de ')]
agente                 recusou=true com texto de resposta: [('nao_consta_12', 'Para conseguir um cão-guia, é necessário se inscrever em um centro de ')]
agente_v2              recusou=true com texto de resposta: [('nao_consta_12', 'Para conseguir um cão-guia, você deve se inscrever em um centro de tre')]
validacao_agente       recusou=true com texto de resposta: [('val_nao_consta_05', 'Não, você não pode trocar de linha sem pagar novamente na Estação Sant')]
validacao_agente_v2    recusou=true com texto de resposta: [('val_nao_consta_05', 'Não, você não pode trocar de linha sem pagar novamente na Estação Sant')]
